# Pricing Experiment A/B Test

An industrial-style experiment analysis notebook focused on decision quality, effect size, and business interpretation.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 5)


from dateutil import parser
def mixed_parse(val):
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    if not s:
        return pd.NaT
    for dayfirst in (False, True):
        try:
            return parser.parse(s, dayfirst=dayfirst)
        except Exception:
            continue
    return pd.NaT


## 2. Load and inspect

In [ ]:
df = pd.read_csv("../data/pricing_experiment.csv")
print(df.shape)
display(df.head())
display(df.isna().mean().sort_values(ascending=False).mul(100).round(2))
print("Duplicate rows:", df.duplicated().sum())


## 3. Clean the experiment data

In [ ]:
def normalize_text(series):
    return (series.astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .replace({"nan": np.nan, "None": np.nan, "": np.nan}))

for col in ["variant","country","customer_segment","device_type","visitor_type"]:
    df[col] = normalize_text(df[col])

df["variant"] = df["variant"].str.lower().replace({"control":"control","treatment":"treatment"})
df["country"] = df["country"].str.upper().fillna("UNKNOWN")
df["customer_segment"] = df["customer_segment"].str.replace("-", "", regex=False).str.title().fillna("Unknown")
df["device_type"] = df["device_type"].str.title()
df["visitor_type"] = df["visitor_type"].str.title().fillna("Unknown")

df["assignment_time"] = df["assignment_time"].apply(mixed_parse)
df["revenue_usd"] = pd.to_numeric(df["revenue_usd"], errors="coerce")
df.loc[df["revenue_usd"] < 0, "revenue_usd"] = np.nan
df["page_load_seconds"] = pd.to_numeric(df["page_load_seconds"], errors="coerce")

df = df.dropna(subset=["user_id", "variant", "assignment_time"]).drop_duplicates(subset=["user_id"], keep="first")
print(df.shape)


## 4. Sample ratio check

In [ ]:
sample_sizes = df["variant"].value_counts()
display(sample_sizes)

observed = sample_sizes.reindex(["control","treatment"]).fillna(0).astype(int).values
expected = np.array([observed.sum()/2, observed.sum()/2])
chi2 = ((observed - expected)**2 / expected).sum()
p_srm = 1 - stats.chi2.cdf(chi2, df=1)
print("SRM p-value:", round(p_srm, 5))


## 5. Primary metric: conversion rate

In [ ]:
conv = df.groupby("variant")["converted"].agg(["sum","count","mean"])
conv["conversion_pct"] = (conv["mean"]*100).round(2)
display(conv)

success = conv["sum"].values
nobs = conv["count"].values
zstat, pval = proportions_ztest(success, nobs)
ci_control = proportion_confint(success[0], nobs[0], alpha=0.05, method="normal")
ci_treatment = proportion_confint(success[1], nobs[1], alpha=0.05, method="normal")

lift = (conv.loc["treatment","mean"] / conv.loc["control","mean"] - 1) * 100
print(f"Conversion lift: {lift:.2f}%")
print(f"p-value: {pval:.5f}")
print("95% CI control:", ci_control)
print("95% CI treatment:", ci_treatment)


## 6. Secondary metric: revenue per visitor

In [ ]:
rpv = df.groupby("variant")["revenue_usd"].mean()
display(rpv)

control_rev = df.loc[df["variant"]=="control", "revenue_usd"].fillna(0)
treat_rev = df.loc[df["variant"]=="treatment", "revenue_usd"].fillna(0)
tstat, pval_rev = stats.ttest_ind(treat_rev, control_rev, equal_var=False)
print("Revenue-per-visitor p-value:", round(pval_rev, 5))


## 7. Segment-level treatment effects

In [ ]:
def segment_lift(data, col):
    g = (data.groupby([col, "variant"])["converted"]
           .mean()
           .unstack())
    g["lift_pct"] = ((g["treatment"] / g["control"]) - 1) * 100
    return g.sort_values("lift_pct", ascending=False).round(4)

display(segment_lift(df, "customer_segment"))
display(segment_lift(df, "device_type"))
display(segment_lift(df, "country").head(10))


## 8. Diagnostic plots

In [ ]:
plot_df = (df.groupby("variant")["converted"].mean()*100).reset_index(name="conversion_pct")
plt.bar(plot_df["variant"], plot_df["conversion_pct"])
plt.title("Conversion rate by variant")
plt.ylabel("Conversion %")
plt.show()

(df.groupby("variant")["page_load_seconds"].mean()
   .plot(kind="bar", title="Average page load by variant"))
plt.ylabel("Seconds")
plt.show()


## 9. Recommendation

In [ ]:
result = {
    "control_conversion_pct": round(conv.loc["control","mean"]*100, 2),
    "treatment_conversion_pct": round(conv.loc["treatment","mean"]*100, 2),
    "relative_lift_pct": round(lift, 2),
    "conversion_p_value": round(float(pval), 5),
    "revenue_p_value": round(float(pval_rev), 5),
    "srm_p_value": round(float(p_srm), 5),
}
result


## 10. Executive summary

Use this section to communicate like a business analyst:
- State whether the experiment passed sample-ratio and data-quality checks.
- Translate the lift into expected business impact.
- Highlight where treatment works best or worst.
- Recommend rollout, holdout, or follow-up experiment.
